In [ ]:
import polars as pl
from fof8_core.loader import FOF8Loader

# 1. Timeline Discovery (to ensure we match the pipeline's logic)
loader = FOF8Loader(base_path="../fof8-gen/data/raw", league_name="DRAFT005")
right_censor_buffer = 20 

valid_start_year = loader.initial_sim_year + 1
valid_end_year = loader.final_sim_year - right_censor_buffer

# 2. Load the "Saved" Features
# This is the file generated by 'uv run dvc repro process_features'
FEATURES_PATH = "../fof8-ml/data/processed/features.parquet"
df = pl.read_parquet(FEATURES_PATH)

# 3. Replicate the Pipeline's Training Split
# The pipeline uses a chronological split where the last X% of valid years is test
test_split_pct = 0.20
total_years = valid_end_year - valid_start_year + 1
test_years_count = int(total_years * test_split_pct)
train_end_year = valid_end_year - test_years_count

# Filter for the Training Set years only
df_train = df.filter(
    (pl.col("Year") >= valid_start_year) & 
    (pl.col("Year") <= train_end_year)
)

# Apply Stage 2 Intensity Filter (Only players who had at least 1 crash/DPO)
df_reg = df_train.filter(pl.col("DPO") > 0)

print(f"Training Years: {valid_start_year} - {train_end_year}")
print(f"Stage 2 Samples: {len(df_reg)}")


In [ ]:
mean_train = np.mean(y_train)
var_train = np.var(y_train)
vmr = var_train / mean_train

print(f"--- Training Set Distribution ---")
print(f"Mean:     {mean_train:.4f}")
print(f"Variance: {var_train:.4f}")
print(f"VMR:      {vmr:.4f}")

if vmr > 1.2:
    print("\nResult: Overdispersion detected. Tweedie is theoretically preferred.")
else:
    print("\nResult: Poisson assumption holds.")


In [ ]:
results = {}
models = {}

for loss in ['Poisson', 'Tweedie:variance_power=1.5']:
    name = loss.split(':')[0]
    print(f"Training {name}...")
    
    model = cb.CatBoostRegressor(
        loss_function=loss,
        iterations=1000,
        early_stopping_rounds=50,
        random_seed=42,
        verbose=False
    )
    
    model.fit(X_train, y_train, eval_set=(X_val, y_val), cat_features=cat_features, use_best_model=True)
    
    models[name] = model

    preds = model.predict(X_val)
    mae = mean_absolute_error(y_val, preds)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    
    results[name] = {
        'MAE': mae, 
        'RMSE': rmse, 
        'eval_result': model.get_evals_result(),
        'best_iteration': model.get_best_iteration()
    }

# Display results
import pandas as pd
print("\n--- Validation Performance ---")
print(pd.DataFrame(results).T[['MAE', 'RMSE']])

In [ ]:
plt.figure(figsize=(12, 6))

for name, data in results.items():
    # 1. Get the loss key for this model (e.g. 'Poisson' or 'Tweedie')
    val_metrics = data['eval_result']['validation']
    loss_key = list(val_metrics.keys())[0]
    
    # 2. Extract both Train (learn) and Validation (validation)
    train_loss = data['eval_result']['learn'][loss_key]
    val_loss = data['eval_result']['validation'][loss_key]
    
    # 3. Plot them (using dashed for train, solid for val)
    plt.plot(train_loss, linestyle='--', label=f'{name} (Train)')
    plt.plot(val_loss, linestyle='-', label=f'{name} (Val)')

plt.title("Convergence Comparison: Poisson vs Tweedie")
plt.xlabel("Iterations")
plt.ylabel("Loss (Negative Log-Likelihood)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
def evaluate_tail(y_true, y_pred, top_pct=0.1):
    threshold = np.quantile(y_true, 1 - top_pct)
    mask = y_true >= threshold
    tail_mae = mean_absolute_error(y_true[mask], y_pred[mask])
    
    # Top-50 Capture
    top_n = 50
    true_top_indices = np.argsort(y_true)[-top_n:]
    pred_top_indices = np.argsort(y_pred)[-top_n:]
    overlap = len(set(true_top_indices).intersection(set(pred_top_indices)))
    capture_rate = overlap / top_n
    
    return tail_mae, capture_rate

print("--- Boom Pick Analysis (Top 10% of Players) ---")
for name, model in models.items():
    preds = model.predict(X_val)
    tail_mae, capture = evaluate_tail(y_val, preds)
    print(f"{name}:")
    print(f"  Tail MAE: {tail_mae:.4f} (Lower is better)")
    print(f"  Top-50 Capture Rate: {capture:.2%} (Higher is better)")
